# Lekcija 17: Kompresija slike &mdash; bez gubitaka i s gubicima

Pohrana slike kao izvornih vrijednosti piksela zahtijeva 1 bajt po kanalu po pikselu, bez obzira na sadržaj slike. Kompresija iskorištava različite vrste redundancije u slikama, odnosno činjenicu da stvarne slike nisu nasumične. Ova lekcija obrađuje i kompresiju **bez gubitaka**, kod koje je dekodirana slika identična izvorniku bit po bit, primjerice PNG, i kompresiju **s gubicima**, kod koje je dekodirana slika samo aproksimacija, primjerice JPEG.


In [ ]:
import numpy as np
import cv2
import heapq
from collections import Counter
import matplotlib.pyplot as plt

## Kompresija bez gubitaka

### Kodiranje duljine nizova (RLE)

Najjednostavniji postupak bez gubitaka: umjesto svakog piksela pohranjujemo parove `(value, run length)`, odnosno vrijednost i duljinu uzastopnog niza jednakih piksela. To je vrlo učinkovito na slikama s velikim jednolikim područjima, poput umjetno izrađene grafike, skeniranog teksta i maski, ali ne daje korisno smanjenje na stvarnim fotografijama.


In [ ]:
def rle_encode(array):
    flat = array.ravel()
    change_points = np.where(np.diff(flat) != 0)[0] + 1
    starts = np.concatenate([[0], change_points])
    ends = np.concatenate([change_points, [len(flat)]])
    return [(flat[s], e - s) for s, e in zip(starts, ends)]

def rle_decode(runs, shape):
    flat = np.concatenate([np.full(length, value) for value, length in runs])
    return flat.reshape(shape).astype(np.uint8)

flat_regions_img = np.zeros((80, 80), dtype=np.uint8)
cv2.rectangle(flat_regions_img, (10, 10), (70, 70), 200, -1)
sp = flat_regions_img.shape

plt.imshow(flat_regions_img, cmap='gray')
plt.title(f'Image with flat regions ({sp[1]}x{sp[0]}) = {sp[0]*sp[1]} bytes')
plt.axis('off')
plt.show()

runs = rle_encode(flat_regions_img)
decoded = rle_decode(runs, flat_regions_img.shape)
print('exact reconstruction?', np.array_equal(decoded, flat_regions_img))

raw_bytes = flat_regions_img.size
rle_bytes = len(runs) * 3  # roughly: 1 byte value + 2 bytes run length, per run
print(f'raw size:  {raw_bytes} bytes')
print(f'RLE size:  ~{rle_bytes} bytes ({raw_bytes / rle_bytes:.1f}x smaller)')

### Huffmanovo kodiranje: manje bitova za česte vrijednosti

Napredniji postupak bez gubitaka je **Huffmanovo kodiranje**, koje koristi činjenicu da se neke vrijednosti piksela, primjerice uobičajena siva boja pozadine, pojavljuju znatno češće od drugih. Izrađuje kod promjenjive duljine u kojem česte vrijednosti dobivaju kratke kodove, a rijetke dulje &mdash; dokazivo optimalan među prefiksnim kodovima.

Shannonova **entropija** daje teorijsku donju granicu prosječnog broja bitova po pikselu za *bilo koji* kod temeljen samo na distribuciji vrijednosti, uz zanemarivanje prostorne strukture:

$$H = -\sum_i p_i \log_2 p_i$$

gdje je $p_i$ vjerojatnost pojave određene vrijednosti piksela u slici, a zbrajanje obuhvaća sve moguće vrijednosti piksela.


In [ ]:
rng = np.random.default_rng(0)
photo_like = np.zeros((100, 100), dtype=np.uint8)
cv2.rectangle(photo_like, (10, 10), (90, 90), 200, -1)
cv2.circle(photo_like, (50, 50), 25, 120, -1)
photo_like = np.clip(photo_like.astype(np.float64) + rng.normal(0, 5, photo_like.shape), 0, 255).astype(np.uint8)
sp = photo_like.shape

counts = Counter(photo_like.ravel().tolist())
total = sum(counts.values())
probs = {value: count / total for value, count in counts.items()}
entropy = -sum(p * np.log2(p) for p in probs.values())


def build_huffman_codes(probs):
    heap = [[p, [symbol, '']] for symbol, p in probs.items()]
    heapq.heapify(heap)
    while len(heap) > 1:
        lo = heapq.heappop(heap)
        hi = heapq.heappop(heap)
        for pair in lo[1:]:
            pair[1] = '0' + pair[1]
        for pair in hi[1:]:
            pair[1] = '1' + pair[1]
        heapq.heappush(heap, [lo[0] + hi[0]] + lo[1:] + hi[1:])
    return {symbol: code for symbol, code in heap[0][1:]}

codes = build_huffman_codes(probs)
avg_bits = sum(probs[symbol] * len(code) for symbol, code in codes.items())

plt.imshow(photo_like, cmap='gray')
plt.title(f'Image with noise ({sp[1]}x{sp[0]}) = {sp[0]*sp[1]} bytes')
plt.axis('off')
plt.show()

print(f'naive fixed-width encoding:           8.00 bits/pixel')
print(f'Shannon entropy (theoretical floor):  {entropy:.2f} bits/pixel')
print(f'Huffman coding achieves:              {avg_bits:.2f} bits/pixel')

Huffmanovo kodiranje približava se granici entropije; točno je doseže samo kada je svaka vjerojatnost potencija broja 2. Stvarni formati bez gubitaka, poput PNG-a, kombiniraju sličnu ideju Huffmanova ili aritmetičkog kodiranja s prethodnim **prediktivnim filtrom** &mdash; svaki piksel predviđa se iz njegovih susjeda i kodira se samo obično mala pogreška predviđanja, koja ima znatno manju entropiju od izvornih vrijednosti piksela.


## Kompresija s gubicima: DCT

Temeljna ideja JPEG-a jest iskoristiti sklonost ljudskog vidnog sustava da zanemari sitne detalje. JPEG dijeli sliku na male blokove veličine 8 × 8, a zatim svaki blok transformira u frekvencijsku domenu kako bi mogao odbaciti visokofrekvencijske komponente koje su ljudskom oku jedva primjetne. JPEG koristi **diskretnu kosinusnu transformaciju (DCT)**, blisku srodnicu Fourierove transformacije iz Lekcije 15. Poput DFT-a, DCT zapisuje blok kao zbroj frekvencijskih komponenti &mdash; ali koristi samo kosinuse, bez imaginarnog dijela, radi izbjegavanja rubnih artefakata.


In [ ]:
block = photo_like[6:14, 6:14].astype(np.float64)  # a block straddling the rectangle's sharp edge
dct_block = cv2.dct(block)

energy = dct_block**2
print(f'fraction of block energy in top-left 2x2 coefficients: {energy[:2, :2].sum() / energy.sum():.2%}')
print(f'fraction of block energy in top-left 4x4 coefficients: {energy[:4, :4].sum() / energy.sum():.2%}')

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(block, cmap='gray')
axes[0].set_title('8x8 pixel block')
axes[1].imshow(np.log1p(np.abs(dct_block)), cmap='gray')
axes[1].set_title('log|DCT coefficients|\n(top-left = low frequency)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

### Kvantizacija: odbacivanje najmanje važnih koeficijenata

Kompresija se provodi **kvantizacijom** DCT koeficijenata &mdash; dijeljenjem određenom vrijednošću i zaokruživanjem. Vrijednosti za dijeljenje uzimaju se iz kvantizacijske tablice oblikovane prema osjetljivosti ljudskog vida. Mnogi koeficijenti, posebno visokofrekvencijski koji nose najmanje energije, postaju točno nula i zauzimaju vrlo malo prostora za pohranu.


In [ ]:
def keep_top_left(dct_block, n):
    """Zero out every coefficient outside the top-left nxn corner (crude stand-in for JPEG-style quantization)."""
    masked = np.zeros_like(dct_block)
    masked[:n, :n] = dct_block[:n, :n]
    return masked

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for ax, n in zip(axes, [8, 4, 2, 1]):
    kept = keep_top_left(dct_block, n)
    reconstructed = cv2.idct(kept)
    nonzero = np.count_nonzero(kept)
    ax.imshow(reconstructed, cmap='gray', vmin=block.min(), vmax=block.max())
    ax.set_title(f'{nonzero}/64 coeffs kept', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Čak i zadržavanje samo gornjeg lijevog dijela veličine 2 × 2, odnosno 4 od 64 koeficijenta &mdash; smanjenje 16 puta &mdash; čuva većinu ukupne strukture bloka; gube se samo najfiniji detalji.


## Stvarni JPEG: kompromis između veličine i izobličenja

Stvarno JPEG kodiranje slijedi isti postupak DCT-a pa kvantizacije za svaki blok, uz pažljivo oblikovanu kvantizacijsku tablicu ovisnu o frekvenciji te RLE i Huffmanovo kodiranje dobivenih rijetkih koeficijenata &mdash; kombinaciju ideja kompresije s gubicima i bez gubitaka iz ove lekcije. Izravno koristimo ugrađeni koder biblioteke OpenCV, mijenjamo postavku kvalitete i mjerimo veličinu datoteke te pogrešku rekonstrukcije.


In [ ]:
def psnr(original, reconstructed):
    mse = np.mean((original.astype(np.float64) - reconstructed.astype(np.float64))**2)
    return float('inf') if mse == 0 else 10 * np.log10(255**2 / mse)

color_img = np.zeros((200, 200, 3), dtype=np.uint8)
cv2.rectangle(color_img, (20, 20), (180, 180), (255, 120, 30), -1)
cv2.circle(color_img, (100, 100), 60, (30, 200, 255), -1)
color_img = np.clip(color_img.astype(np.float64) + rng.normal(0, 8, color_img.shape), 0, 255).astype(np.uint8)
sp = color_img.shape

qualities = [10, 30, 50, 80, 95, 100]
sizes, psnrs = [], []
for q in qualities:
    ok, encoded = cv2.imencode('.jpg', color_img, [cv2.IMWRITE_JPEG_QUALITY, q])
    decoded = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
    sizes.append(len(encoded))
    psnrs.append(psnr(color_img, decoded))

plt.imshow(color_img)
plt.title(f'Image with noise ({sp[1]}x{sp[0]}) = {sp[0]*sp[1]*sp[2]} bytes')
plt.axis('off')
plt.show()

raw_size = color_img.size
print(f'{"quality":>8} {"file size (bytes)":>18} {"compression ratio":>18} {"PSNR (dB)":>10}')
for q, s, p in zip(qualities, sizes, psnrs):
    print(f'{q:>8} {s:>18} {raw_size / s:>17.1f}x {p:>10.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(qualities, sizes, marker='o')
axes[0].set_xlabel('JPEG quality')
axes[0].set_ylabel('file size (bytes)')
axes[0].set_title('Size vs. quality')

axes[1].plot(sizes, psnrs, marker='o')
axes[1].set_xlabel('file size (bytes)')
axes[1].set_ylabel('PSNR (dB)')
axes[1].set_title('Rate-distortion curve')
plt.tight_layout()
plt.show()

To je temeljni kompromis kompresije s gubicima: svaki dodatni bajt koji ste spremni pohraniti daje sve manji dobitak u kvaliteti, odnosno krivulja odnosa veličine i izobličenja pri visokoj kvaliteti postaje sve ravnija. Nema besplatnog poboljšanja, nego samo odabira položaja na krivulji. Za tipičnu fotografiju JPEG kvaliteta 75 neće biti primjetna bez povećavanja prikaza, a smanjit će veličinu datoteke 6–8 puta.


## JPEG ili PNG? Odabir odgovarajućeg formata

Prethodni kompromisi daju praktično pravilo:

- Koristite **JPEG za fotografije**. Na fotografijama prevladavaju glatki prijelazi &mdash; upravo ono što DCT koncentrira u nekoliko niskofrekvencijskih koeficijenata, pa je odbacivanje ostalih jedva vidljivo. JPEG tako daje veliko smanjenje veličine uz mali perceptivni gubitak.

- Koristite **PNG za grafiku**: snimke zaslona, dijagrame, tekst, ikone i sve što ima jednolika područja ili oštre rubove. Grafika je suprotan slučaj: oštar rub raspoređuje energiju preko *svih* DCT frekvencija, pa ista kvantizacija koju fotografije podnose stvara vidljive oscilacije i blokovske artefakte oko rubova i teksta. Kompresija bez gubitaka učinkovito obrađuje jednolika područja i oštre rubove, upravo ono u čemu su dobri RLE i prethodni prediktivni filtar, pa je PNG za takav sadržaj često i bez artefakata *i* manji od JPEG-a.


In [ ]:
photo = cv2.cvtColor(cv2.imread('../img/house.png'), cv2.COLOR_BGR2RGB)

graphic = np.full((100, 200, 3), 255, dtype=np.uint8)
cv2.rectangle(graphic, (20, 20), (180, 100), (40, 40, 40), -1)
cv2.putText(graphic, 'REPORT', (30, 70), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 2)
cv2.line(graphic, (30, 80), (160, 80), (255, 255, 255), 2)

def encode_sizes(image, jpeg_quality=75):
    ok, jpg = cv2.imencode('.jpg', image, [cv2.IMWRITE_JPEG_QUALITY, jpeg_quality])
    ok, png = cv2.imencode('.png', image, [cv2.IMWRITE_PNG_COMPRESSION, 9])
    return jpg, png

jpg_photo, png_photo = encode_sizes(photo)
jpg_graphic, png_graphic = encode_sizes(graphic)

print(f'{"":12} {"Original":>11} {"PNG":>11} {"JPEG (q75)":>11}')
print(f'{"photograph":12} {int(np.prod(photo.shape)):>9} B {len(png_photo):>9} B {len(jpg_photo):>9} B    (PNG is bigger than JPEG by {len(png_photo)/len(jpg_photo):.1f}x)')
print(f'{"graphic":12} {int(np.prod(graphic.shape)):>9} B {len(png_graphic):>9} B {len(jpg_graphic):>9} B    (PNG is smaller than JPEG by {len(png_graphic)/len(jpg_graphic):.1f}x)')

fig, axes = plt.subplots(2, 2, figsize=(7, 7))
axes[0, 0].imshow(photo); axes[0, 0].set_title(f'Photo: original ({int(np.prod(photo.shape))} B)')
axes[0, 1].imshow(cv2.imdecode(jpg_photo, cv2.IMREAD_COLOR)); axes[0, 1].set_title(f'Photo: JPEG q75 ({len(jpg_photo)} B)')
axes[1, 0].imshow(graphic); axes[1, 0].set_title(f'Graphic: original ({int(np.prod(graphic.shape))} B)')
axes[1, 1].imshow(cv2.imdecode(jpg_graphic, cv2.IMREAD_COLOR)); axes[1, 1].set_title(f'Graphic: JPEG q75 ({len(jpg_graphic)} B)')
for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()
plt.show()

<p class="image-attribution">Autor fotografije: <a href="https://unsplash.com/@tama66" target="_blank" rel="noopener">Peter Herrmann</a>, na platformi <a href="https://unsplash.com/photos/brown-and-white-concrete-building-eZaEWy2rAIc" target="_blank" rel="noopener">Unsplash</a></p>


### Detaljniji pogled na JPEG artefakte

Povećavanjem područja visokog kontrasta, poput obrisa krova nasuprot nebu, pri znatno nižoj postavci kvalitete izravno vidimo cijenu agresivne kvantizacije: kvadratne blokove veličine 8 × 8 i oscilacije oko oštrog ruba, upravo prethodno opisane nedostatke.


In [ ]:
y0, y1, x0, x1 = 10, 60, 50, 150  # roofline against the sky

low_q = 10
jpg_low_q, _ = encode_sizes(photo, jpeg_quality=low_q)
decoded_low_q = cv2.imdecode(jpg_low_q, cv2.IMREAD_COLOR)

zoom = 4
original_crop = cv2.resize(photo[y0:y1, x0:x1], None, fx=zoom, fy=zoom, interpolation=cv2.INTER_NEAREST)
jpeg_crop = cv2.resize(decoded_low_q[y0:y1, x0:x1], None, fx=zoom, fy=zoom, interpolation=cv2.INTER_NEAREST)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(original_crop)
axes[0].set_title('Original crop (zoomed 4x)')
axes[1].imshow(jpeg_crop)
axes[1].set_title(f'Same crop, JPEG q{low_q}\n(full image: {len(jpg_low_q)} B)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

### Zadaci

1. Pokrenite RLE pokus na `photo_like`, koji sadrži Gaussov šum, umjesto na `flat_regions_img`. Koliko nizova daje i što to govori o prikladnosti RLE-a za šumne prirodne slike?
2. Funkcija `keep_top_left` namjerno je gruba zamjena za stvarnu kvantizaciju, koja postupno smanjuje *sve* koeficijente umjesto izravnog postavljanja dijela njih na nulu. Zamijenite je izrazom `np.round(dct_block / step) * step` za nekoliko različitih vrijednosti `step` i usporedite vidljive artefakte s inačicom koja zadržava gornji lijevi dio.
3. Pri vrlo niskoj JPEG kvaliteti može se pojaviti vidljiv mrežasti uzorak poravnat s granicama blokova veličine 8 × 8, odnosno blokovski artefakti. S obzirom na vezu granica blokova i DCT-a, objasnite zašto bi kvantizacija svakog bloka *neovisno* stvorila upravo takav artefakt.
